In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import boto3
import faiss
import pickle

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.embeddings.embedder import Embedding


In [2]:
pwd

'/Users/akosua/Desktop/Study/AWS_Practice/research_assistant/notebooks'

In [9]:
# download and read index and metadata

BUCKET = "researchassistant-rag-prod-embeddings"
INDEX_PATH = "claims_test.index"
META_PATH = "meta_test.pkl"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"

session = boto3.Session(profile_name="aws_learning")
s3 = session.client("s3", region_name="ca-central-1")

downloaded_index_path = "downloads/faiss_claims.index"
downloaded_metadata_path = "downloads/metadata.pkl"

s3.download_file(BUCKET, Key = "faiss/claims.index", Filename= downloaded_index_path )
s3.download_file(BUCKET, "faiss/claims_meta.pkl", downloaded_metadata_path)

downloaded_index = faiss.read_index(downloaded_index_path)
with open(downloaded_metadata_path, "rb") as f:
    downloaded_metadata = pickle.load(f)

In [10]:
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"
session = boto3.Session(profile_name="aws_learning")
bedrock_client = session.client(
    service_name="bedrock-runtime",
    region_name="ca-central-1",
)
user_query = "fasting"
embedder = Embedding(EMBED_MODEL_ID, bedrock_client)
embeddings = embedder.get_embedder(text=user_query, dimensions=256)

In [11]:
user_embeddings = np.array(embeddings, dtype="float32").reshape(1, -1)
user_embeddings.shape

(1, 256)

In [12]:
scores, indices = downloaded_index.search(user_embeddings, 3)

In [15]:
indices

array([[6, 7, 1]])

In [16]:
scores

array([[0.15112162, 0.14141172, 0.12043948]], dtype=float32)

In [17]:
downloaded_metadata[0].copy()

{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::001',
 'embedding_text': "Claim: The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group Evidence: ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.']",
 'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
  'claim': 'The reduction of systolic blood pressure was 5.595 ± 4.072 mm Hg in the DASH group and 8.459 ± 4.260 mm Hg in the DASH + TRE group',
  'evidence': ['The reduction of systolic blood pressure and diastolic blood pressure were 5.595 ± 4.072 and 5.351 ± 5.643 mm Hg in the DASH group and 8.459 ± 4.260 and 9.459 ± 4.375 mm Hg in the DASH + TRE group.'],
  'section': ['Abstract - Results'],
  'study_quality': {'study_design': 'rct',
   'sample_size': 'small',
   'bias_risk': 'moderate',
   'e

In [13]:
results = []
for idx, score in zip(indices[0], scores[0]):
    item = downloaded_metadata[idx].copy()
    item["similarity_score"] = float(score)
    results.append(item)
results

[{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
  'embedding_text': "Claim: Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group Evidence: ['PSQI before and after intervention in DASH group, n=26:26, p >0.05']",
  'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
   'claim': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group',
   'evidence': ['PSQI before and after intervention in DASH group, n=26:26, p >0.05'],
   'section': ['Supplementary Material 2'],
   'study_quality': {'study_design': 'rct',
    'sample_size': 'small',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence stren

In [25]:
results = [{**x, "source":"retrieved"} for x in results]
results

[{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
  'embedding_text': "Claim: Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group Evidence: ['PSQI before and after intervention in DASH group, n=26:26, p >0.05']",
  'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
   'claim': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group',
   'evidence': ['PSQI before and after intervention in DASH group, n=26:26, p >0.05'],
   'section': ['Supplementary Material 2'],
   'study_quality': {'study_design': 'rct',
    'sample_size': 'small',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence stren

In [ ]:
ClaimEvidence = {
    # Identity
    "claim_id": str,              # paper_id_claim_XXX (stable)
    "paper_id": str,
    "title": str,
    "year": int,

    # Core content (LLM-visible)
    "claim": str,
    "evidence": str,
    "section": str,

    # Quality signals (machine + LLM)
    "study_quality": {
        "overall": "high | moderate | low | unclear",
        "study_type": str,         # RCT, observational, animal, in-vitro
        "sample_size": int | None,
        "notes": str               # short, trimmed
    },

    # Retrieval metadata (planner-visible, summarizer-visible)
    "retrieval": {
        "source": "fresh | faiss",
        "similarity_score": float | None,
        "rank": int | None
    },

    # Optional (future-safe)
    "fields_of_study": List[str] | None
}

#### merge fresh claims with retrieved

In [8]:
from src.pipelines.end_to_end_index import test

In [14]:
fresh_claims = test(user_query, 4)

 Extracted keywords: fasting, metabolic response, autophagy
Number of papers: 4
Error downloading PDF: 403 Client Error: Forbidden for url: https://journals.physiology.org/doi/pdf/10.1152/ajpheart.00041.2016
Error processing paper 284645bcd4331cebd35166761e990ae3c6836a44: An error occurred while processing the PDF: 284645bcd4331cebd35166761e990ae3c6836a44. Error: 'NoneType' object has no attribute 'splitlines'
Error downloading PDF: 403 Client Error: Forbidden for url: https://www.science.org/doi/pdf/10.1126/science.abc4203?download=true
Error processing paper be2bfd6c1c63d5d56d614bcbd62b6204aa038681: An error occurred while processing the PDF: be2bfd6c1c63d5d56d614bcbd62b6204aa038681. Error: 'NoneType' object has no attribute 'splitlines'
Error downloading PDF: 403 Client Error: Forbidden for url: https://www.tandfonline.com/doi/pdf/10.1080/15548627.2020.1788889?needAccess=true
Error processing paper 53f6aa48de6d296acdc54990ef8dfb7bd7f56880: An error occurred while processing the PDF:

In [15]:
fresh_claims

[[{'id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
   'embedding_text': "Claim: Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii Evidence: ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsive than the muscle to starvation-induced autophagy.']",
   'metadata': {'paper_id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e',
    'claim': 'Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii',
    'evidence': ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more res

In [16]:
def passes_quality_gate(claim):
    sq = claim.get("metadata", {}).get("study_quality")

    if not sq:
        return False

    if sq.get("study_design") not in {
        "rct", "systematic_review", "meta_analysis", "controlled_trial"
    }:
        return False

    if sq.get("bias_risk") not in {"low", "moderate"}:
        return False

    return True

In [24]:
passed = []
rejected = []

for x in results:
    if passes_quality_gate(x):
        passed.append(x)
    else:
        rejected.append(x)

In [23]:
results_filtered = [x for x in results if passes_quality_gate(x)]
results_filtered

[{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
  'embedding_text': "Claim: Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group Evidence: ['PSQI before and after intervention in DASH group, n=26:26, p >0.05']",
  'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
   'claim': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group',
   'evidence': ['PSQI before and after intervention in DASH group, n=26:26, p >0.05'],
   'section': ['Supplementary Material 2'],
   'study_quality': {'study_design': 'rct',
    'sample_size': 'small',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence stren

## TESTS - pytest

In [ ]:
def make_claim(
    study_design="rct",
    bias_risk="low",
    evidence_strength="strong",
    sample_size="small"
):
    return {
        "metadata": {
            "study_quality": {
                "study_design": study_design,
                "bias_risk": bias_risk,
                "evidence_strength": evidence_strength,
                "sample_size": sample_size,
            }
        }
    }



# small RCT should pass
def test_small_rct_passes():
    claim = make_claim(
        study_design="rct",
        bias_risk="moderate",
        sample_size="small"
    )

    assert passes_quality_gate(claim) is True


# observational study design should fail
def test_observational_fails():
    claim = make_claim(
        study_design="observational",
        bias_risk="low"
    )

    assert passes_quality_gate(claim) is False


# high bias should fail even if rct
def test_high_bias_fails():
    claim = make_claim(
        study_design="rct",
        bias_risk="high"
    )

    assert passes_quality_gate(claim) is False

In [17]:
# soft scoring assuming all claims pass strict quality gate

def soft_score(claim):
    sim = claim["similarity_score"]  # 0–1 cosine
    sq = claim["metadata"]["study_quality"]

    design_bonus = {
        "meta_analysis": 1.0,
        "systematic_review": 0.9,
        "rct": 0.8,
        "controlled_trial": 0.7,
    }.get(sq["study_design"], 0.0)

    bias_penalty = {
        "low": 1.0,
        "moderate": 0.85
    }.get(sq["bias_risk"], 0.0)

    return (
        0.5 * sim +
        0.3 * design_bonus +
        0.2 * bias_penalty
    )

ranked = sorted(
    results_filtered,
    key=soft_score,
    reverse=True
)
ranked

NameError: name 'results_filtered' is not defined

In [18]:
# needed for fresh claims to be passed through passes_quality_gate
def normalize_similarity(claim, default_sim=0.6):
    if "similarity_score" not in claim or claim["similarity_score"] is None:
        claim["similarity_score"] = default_sim
    return claim


def merge_and_rank_claims(
    retrieved_claims,
    fresh_claims,
    passes_quality_gate
):
    all_claims = []

    for c in retrieved_claims + fresh_claims:
        if not passes_quality_gate(c):
            continue
        all_claims.append(normalize_similarity(c))

    return sorted(all_claims, key=soft_score, reverse=True)

In [28]:
results

[{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
  'embedding_text': "Claim: Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group Evidence: ['PSQI before and after intervention in DASH group, n=26:26, p >0.05']",
  'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
   'claim': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group',
   'evidence': ['PSQI before and after intervention in DASH group, n=26:26, p >0.05'],
   'section': ['Supplementary Material 2'],
   'study_quality': {'study_design': 'rct',
    'sample_size': 'small',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence stren

In [19]:
flattened_claims = [item for sublist in fresh_claims for item in sublist]
flattened_claims

[{'id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
  'embedding_text': "Claim: Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii Evidence: ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsive than the muscle to starvation-induced autophagy.']",
  'metadata': {'paper_id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e',
   'claim': 'Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii',
   'evidence': ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsi

In [20]:
len(flattened_claims)

10

In [26]:
flattened_claims = [{**x, "source":"fresh"} for x in flattened_claims]
flattened_claims

[{'id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
  'embedding_text': "Claim: Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii Evidence: ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsive than the muscle to starvation-induced autophagy.']",
  'metadata': {'paper_id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e',
   'claim': 'Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii',
   'evidence': ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsi

In [27]:
merged_claims = merge_and_rank_claims(results, flattened_claims, passes_quality_gate)

In [28]:
len(merged_claims)

3

In [29]:
merged_claims

[{'id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
  'embedding_text': "Claim: Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group Evidence: ['PSQI before and after intervention in DASH group, n=26:26, p >0.05']",
  'metadata': {'paper_id': '9aff0461b7e1ecceef4f4b97027a83bda7163dbe',
   'claim': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in the DASH group',
   'evidence': ['PSQI before and after intervention in DASH group, n=26:26, p >0.05'],
   'section': ['Supplementary Material 2'],
   'study_quality': {'study_design': 'rct',
    'sample_size': 'small',
    'bias_risk': 'moderate',
    'evidence_strength': 'strong',
    'notes': 'Study is explicitly described as a randomized controlled trial with two intervention groups (DASH and DASH+TRE). Sample sizes appear modest (n=26 for DASH group, n=37 for DASH+TRE group, total n=63), classified as small. Evidence stren

### citation-aware summarization agent

In [ ]:
from src.agents.summarizer import SummarizeAgent
summarizer = SummarizeAgent()
summarized = summarizer.run_summarizer(merged_claims)

In [36]:
from src.preprocessing.clean_json import clean_json
cleaned_summarized = clean_json(summarized)
cleaned_summarized

{'summary': [{'statement': 'Both DASH and DASH+TRE interventions reduced diastolic blood pressure, with DASH+TRE showing greater reduction (9.459 ± 4.375 mm Hg) compared to DASH alone (5.351 ± 5.643 mm Hg).',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::002']},
  {'statement': 'Pittsburgh Sleep Quality Index (PSQI) showed no significant change before and after intervention in either the DASH group (n=26, p>0.05) or the DASH+TRE group (n=37, p>0.05).',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::007',
    '9aff0461b7e1ecceef4f4b97027a83bda7163dbe::008']}],
 'confidence_note': 'Evidence is derived from a single randomized controlled trial with small sample size (total n=63) and moderate bias risk due to unclear blinding status and lack of reported allocation concealment details. Strong evidence strength is based on specific numerical outcomes with standard deviations and statistical significance testing.'}

In [54]:

from src.pipelines.inference import run_inference

In [55]:
res = run_inference("insulin sensitivity and glucose uptake")

 Extracted keywords: insulin sensitivity, glucose uptake, metabolic regulation
Number of papers: 10
Error downloading PDF: 403 Client Error: Forbidden for url: https://www.mdpi.com/1422-0067/21/6/1949/pdf?version=1584266274
Error processing paper 8e96e0eda82b871e998f40ac13ba8a2a96aab4c6: An error occurred while processing the PDF: 8e96e0eda82b871e998f40ac13ba8a2a96aab4c6. Error: 'NoneType' object has no attribute 'splitlines'
Error downloading PDF: 403 Client Error: Forbidden for url: https://academic.oup.com/endo/article-pdf/150/9/4084/9006008/endo4084.pdf
Error processing paper 411856ace694b6f9915a73762f09f62c084508cc: An error occurred while processing the PDF: 411856ace694b6f9915a73762f09f62c084508cc. Error: 'NoneType' object has no attribute 'splitlines'
Error downloading PDF: 403 Client Error: Forbidden for url: https://www.tandfonline.com/doi/pdf/10.1080/21623945.2015.1122856?needAccess=true
Error processing paper 3c740354ccc8d6a2f3ef8daeee159170fc4ad442: An error occurred while

In [56]:
res

{'summary': [{'statement': 'The DASH diet combined with time-restricted eating (DASH + TRE) resulted in a marked increase in night urinary sodium excretion compared to DASH alone, with 24-hour urinary sodium excretion also increased in the DASH + TRE group after 6 weeks.',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::005']},
  {'statement': 'Life Events Score showed no significant changes before and after intervention in either the DASH group or the DASH + TRE group, suggesting that psychosocial stress levels were not meaningfully altered by either intervention.',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::009']}],
 'confidence_note': 'Evidence is derived from a single randomized controlled trial with small sample size (n=63 total) and moderate bias risk due to unclear blinding status and lack of reported allocation concealment details. Strong evidence strength is based on specific numerical outcomes with statistical significance te

### counterargument

In [79]:
len(results)

3

In [60]:
len(flattened_claims)

10

In [64]:
normalized_claims = [normalize_similarity(c) for c in flattened_claims]
normalized_claims

[{'id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
  'embedding_text': "Claim: Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii Evidence: ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsive than the muscle to starvation-induced autophagy.']",
  'metadata': {'paper_id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e',
   'claim': 'Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii',
   'evidence': ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsi

In [65]:
all_claims = results + normalized_claims
len(all_claims)

13

In [62]:
def select_counter_claims(claims, k=5):
    """
    Select lower-ranked but still high-quality claims
    to surface limitations or contradictory evidence.
    """
    eligible = [
        c for c in claims
        if c["metadata"]["study_quality"]["evidence_strength"] in {"strong", "moderate"}
    ]

    # Lower relevance, not lower quality
    eligible = sorted(eligible, key=soft_score)

    return eligible[:k]

In [66]:
selected_claims = select_counter_claims(all_claims)
selected_claims

[{'id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
  'embedding_text': "Claim: Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii Evidence: ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsive than the muscle to starvation-induced autophagy.']",
  'metadata': {'paper_id': 'c27f6eaf3493ad9adc55ed764690dff220d7424e',
   'claim': 'Starvation up-regulated the expression of the autophagy marker LC3 in both hepatopancreas and muscle tissues of male Macrobrachium rosenbergii',
   'evidence': ['Starvation up-regulated the expression of the autophagy marker LC3 in both tissues. Yet, based on the relative levels of the autophagosome-associated LC3-II isoform and of its precursor LC3-I, the hepatopancreas was more responsi

In [68]:
import boto3
#from research_assistant.src.preprocessing.clean_json import clean_json
from src.preprocessing.clean_json import clean_json

BRT = boto3.client("bedrock-runtime")
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"



BRT = boto3.client("bedrock-runtime")
SYSTEM_PROMPT = (
    "You are a scientific research analyst.\n\n"
    "Your task is to identify limitations, null results, or contradictory findings based only on the provided claims.\n"
    "\n"
    "You must:\n"
    "- Use ONLY the supplied claims\n"
    "- Do NOT restate the main summary\n"
    "- Highlight where evidence is weak, mixed, or context-dependent\n"
    "- Cite the claim IDs for every statement\n"
    "- Avoid speculative language\n"
    "- If no meaningful limitations exist, state that explicitly.\n\n"
    
    "Output rules(STRICT):\n"
    "- Output must follow the exact JSON schema provided:\n\n"
    "{\n"
    '"limitations": [\n'
    '    {\n'
    '    "statement": "Some trials report no significant improvement in glycemic control among patients with long-standing type 2 diabetes.",\n'
    '        "supporting_claim_ids": [\n'
    '            "paperA::003",\n'
    '        ]\n'
    '        },\n'
    '        {\n'
    '        "statement": "Some studies report limited clinical impact of AMPK activation in advanced type 2 diabetes.",\n'
    '        "supporting_claim_ids": [\n'
    '            "paperC::005"\n'
    '        ]\n'
    '        }\n'
    '    ],\n'
    "    }"
)




class CounterArgumentAgent:
    def __init__(self, model_id: str = "global.anthropic.claude-haiku-4-5-20251001-v1:0", max_output_tokens: int = 900):
        self.model_id = model_id
        self.max_output_tokens = max_output_tokens


    def build_user_prompt(self, text) -> str:
        return f"""
    You are given a list of empirical claims from research papers.
    Text:
    \"\"\"
    {text}
    \"\"\"
    Your task is to identify limitations, null results, or contradictory findings based only on the provided claims.
    """.strip()


    def run_counterarg(self, claims:list):
        conversation = [
            {
                "role": "user",
                "content": [{"text": self.build_user_prompt(claims)}],
            },
        ]

        response = BRT.converse(
            modelId=self.model_id,
            system=[
                {
                    "text": SYSTEM_PROMPT
                }
            ],
            messages=conversation,
            inferenceConfig={
                "maxTokens": self.max_output_tokens,
                "temperature": 0.0,
            },
        )

        stop_reason = response.get("stopReason", "")
        if stop_reason == "max_tokens":
            print(f"Output truncated")
            return[]

        output_text = response["output"]["message"]["content"][0]["text"]

        return output_text


In [80]:
counter = CounterArgumentAgent()
counter_res = counter.run_counterarg(selected_claims)

SSLError: SSL validation failed for https://bedrock-runtime.ca-central-1.amazonaws.com/model/global.anthropic.claude-haiku-4-5-20251001-v1%3A0/converse [Errno 2] No such file or directory

In [ ]:
counter_res = clean_json(counter_res)
counter_res 

{'limitations': [{'statement': 'Study design and sample size cannot be determined from the provided claims and metadata, limiting assessment of methodological rigor and generalizability.',
   'supporting_claim_ids': ['c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::002',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::003',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::004',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::005']},
  {'statement': 'Moderate bias risk exists due to lack of information about randomization, blinding, control group design, and potential confounding variables across all reported findings.',
   'supporting_claim_ids': ['c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::002',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::003',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::004',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::005']},
  {'statement': 'Muscle tissue showed 

In [ ]:
def build_final_response(summary_output, counter_output):
    key_findings = summary_output["summary"]
    limitations = counter_output.get("limitations", [])

    citation_ids = set()
    for item in key_findings + limitations:
        citation_ids.update(item.get("supporting_claim_ids", []))

    return {
        "query":user_query,
        "answer": {
            "key_findings": key_findings,
            "confidence_note": summary_output.get("confidence_note", ""),
            "evidence_strength": 
        },
        "limitations": limitations,
        "metadata": {
            "num_supporting_claims": len(citation_ids),
            "num_limitations": len(limitations),
            "citation_ids": sorted(citation_ids)
        }
    }

In [74]:
res

{'summary': [{'statement': 'The DASH diet combined with time-restricted eating (DASH + TRE) resulted in a marked increase in night urinary sodium excretion compared to DASH alone, with 24-hour urinary sodium excretion also increased in the DASH + TRE group after 6 weeks.',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::005']},
  {'statement': 'Life Events Score showed no significant changes before and after intervention in either the DASH group or the DASH + TRE group, suggesting that psychosocial stress levels were not meaningfully altered by either intervention.',
   'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::009']}],
 'confidence_note': 'Evidence is derived from a single randomized controlled trial with small sample size (n=63 total) and moderate bias risk due to unclear blinding status and lack of reported allocation concealment details. Strong evidence strength is based on specific numerical outcomes with statistical significance te

In [76]:
counter_res

{'limitations': [{'statement': 'Study design and sample size cannot be determined from the provided claims and metadata, limiting assessment of methodological rigor and generalizability.',
   'supporting_claim_ids': ['c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::002',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::003',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::004',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::005']},
  {'statement': 'Moderate bias risk exists due to lack of information about randomization, blinding, control group design, and potential confounding variables across all reported findings.',
   'supporting_claim_ids': ['c27f6eaf3493ad9adc55ed764690dff220d7424e::001',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::002',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::003',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::004',
    'c27f6eaf3493ad9adc55ed764690dff220d7424e::005']},
  {'statement': 'Muscle tissue showed 

In [77]:
final = build_final_response(res, counter_res)
final

{'answer': {'key_findings': [{'statement': 'The DASH diet combined with time-restricted eating (DASH + TRE) resulted in a marked increase in night urinary sodium excretion compared to DASH alone, with 24-hour urinary sodium excretion also increased in the DASH + TRE group after 6 weeks.',
    'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::005']},
   {'statement': 'Life Events Score showed no significant changes before and after intervention in either the DASH group or the DASH + TRE group, suggesting that psychosocial stress levels were not meaningfully altered by either intervention.',
    'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::009']}],
  'confidence_note': 'Evidence is derived from a single randomized controlled trial with small sample size (n=63 total) and moderate bias risk due to unclear blinding status and lack of reported allocation concealment details. Strong evidence strength is based on specific numerical outcomes with statist

In [78]:
final

{'answer': {'key_findings': [{'statement': 'The DASH diet combined with time-restricted eating (DASH + TRE) resulted in a marked increase in night urinary sodium excretion compared to DASH alone, with 24-hour urinary sodium excretion also increased in the DASH + TRE group after 6 weeks.',
    'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::005']},
   {'statement': 'Life Events Score showed no significant changes before and after intervention in either the DASH group or the DASH + TRE group, suggesting that psychosocial stress levels were not meaningfully altered by either intervention.',
    'supporting_claim_ids': ['9aff0461b7e1ecceef4f4b97027a83bda7163dbe::009']}],
  'confidence_note': 'Evidence is derived from a single randomized controlled trial with small sample size (n=63 total) and moderate bias risk due to unclear blinding status and lack of reported allocation concealment details. Strong evidence strength is based on specific numerical outcomes with statist